# Example Notebook

`data/raw/` holds source data exactly as it arrived — nothing in this project ever
edits it in place. This notebook reads `data/raw/items.csv` and loads it into the
`Items` repository (backed by `app.db`), the same starting point a real ingestion
task would use. Anything derived — cleaned, joined, aggregated — belongs in
`data/processed/`, not back in `raw/`.

## Using a Repository standalone

`@Repository` returns the decorated class unchanged — no base class, no injected
methods, no hidden state. It only registers the class and its `Entity` relationship
so the `Context` *can* wire it up inside a running `Project`. Nothing stops you from
instantiating it directly and passing its dependencies by hand, exactly like any
other Python object — which is what a notebook kernel needs, since it runs outside
the resolved object graph.

Here `Items` (`app/repositories.py`) normally receives its `SQLite` dependency via
DI. Standalone, we build that same dependency chain ourselves: a `Config` with just
the bit `SQLite` needs, then `SQLite(config)`, then `Items(sqlite)`.

In [ ]:
import sys

sys.path.insert(0, "..")

from app.entities import Item
from app.repositories import Items
from app.storages import SQLite
from opendataframework import Config

config = Config({"sqlite": {"path": "../app.db"}})
sqlite = SQLite(config)
items = Items(sqlite)

In [ ]:
import csv

with open("../data/raw/items.csv", newline="") as f:
    for row in csv.DictReader(f):
        items.save(Item(id=None, name=row["name"], quantity=int(row["quantity"])))

items.all()